# 08 — Allocation (`EXP-H06`)

## 1. Objective
Test whether a pre-registered opportunity/risk allocator improves after-cost Sharpe relative to equal-notional and volatility-targeted sizing under one frozen H5 ES budget.

## 2. Hypothesis
At each decision, RORAC equals the direction-specific expected barrier payoff divided by the point-in-time unit ES. Size is the signed gross cap times that ratio divided by its training-only 90th percentile, clipped to [0, 1]. Under the retained H5 budget, this beats both controls at 5, 15, and 60 minutes.

## 3. Null
RORAC does not beat both controls in after-cost Sharpe at every horizon after Holm correction across the two control comparisons, or it fails the common ES budget.

## 4. Information set
BTCUSDT one-minute OHLC known at each decision close. The H4 result supplies the frozen direction. The sealed retained H5 result supplies the single budget policy. H6 may change size only.

## 5. Dataset
`btcusdt-1m`, bound to the same frozen manifest as H4 and H5.

## 6. Data quality
The standard minute-bar audit applies. Missing risk estimates, denominator floors, gross caps, ES binds, and failed origins are counted.

## 7. Labels
Opportunity is barrier width times target-minus-stop probability, minus the frozen round-trip cost, floored at zero. Risk capital is the training 97.5% ES of a unit position scaled by the causal volatility ratio. Economic outcomes are after-cost one-minute returns.

## 8. Baseline
Equal-notional and volatility-targeted sizing are the nested controls. Both pass through the same H5 budget as RORAC. Volatility-targeted sizing remains the default until H6 is retained.

## 9. Model family
The registered ladder is `equal_notional`, `volatility_targeted`, and `rorac`. The candidate is `rorac`. There is no second budget arm.

## 10. Splits
24-month train, 6-month validation, 6-month step, and final 12-month untouched test.

## 11. Configuration
The source of truth is `research.protocol.H6_CONFIG`. Development inspects the H5 record and returns a deferral without scoring until that budget is sealed and retained. Sealed H6 confirmation requires the retained budget and a development result of `rorac`.

In [1]:
import os
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while root != root.parent and not (root / "research" / "__init__.py").exists():
    root = root.parent
if not (root / "research" / "__init__.py").exists():
    raise RuntimeError("open this notebook from the trading repository")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ.setdefault("AWS_PROFILE", "trading")
os.environ.setdefault("AWS_DEFAULT_REGION", "ap-southeast-2")
os.environ.setdefault("DATA_STORE", "s3")
os.environ.setdefault("S3_BUCKET", "trading-data-centheos")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.6g}".format)

from research.registry import config_digest, load_record
from research.report import (
    allocation_summary,
    confirmation_blockers,
    plot_allocation_activity,
    plot_allocation_sharpe,
    quality_table,
    record_development_decision,
)

## 5–6. Dataset and data quality

Reload and verify the frozen single-series manifest. Collector rows appended after the freeze are counted and excluded.

In [2]:
from research.evaluate import align_to_frozen_dataset
from research.load import load_research_bars
from research.manifest import load_manifest
from research.protocol import H6_CONFIG

MANIFEST_PATH = root / "research" / "registry" / "datasets" / "btcusdt-1m.json"
manifest = load_manifest(MANIFEST_PATH)
loaded_bars, objects = load_research_bars(
    venue=H6_CONFIG["venue"],
    symbol=H6_CONFIG["symbol"],
    timeframe=H6_CONFIG["timeframe"],
    bucket=H6_CONFIG["bucket"],
    profile=H6_CONFIG["profile"],
    region=H6_CONFIG["region"],
)
bars, quality, collector_rows_after_freeze = align_to_frozen_dataset(
    loaded_bars, manifest, objects
)
print("dataset_id:", manifest.dataset_id)
print("dataset_digest:", manifest.digest)
print("frozen_window:", manifest.start, "to", manifest.end)
print("collector_rows_after_freeze:", collector_rows_after_freeze)
display(quality_table({H6_CONFIG["symbol"]: quality}))

Excluded 6987 collector rows after frozen end 2026-10-17T10:43:00+00:00; ohlcv/binance/BTCUSDT/1m/2026.parquet etag b67bbf4dddd1bcb2658bac4d361452ec -> 02765ddf77c7adf892c6fd3e1655d56d, size 5482639 -> 5663899


dataset_id: btcusdt-1m
dataset_digest: c1192eca8fa673c386b4ff3aad751f4f2d4895e2b02d7b693a42aac1ecd28188
frozen_window: 2020-01-01T00:00:00+00:00 to 2026-10-17T10:43:00+00:00
collector_rows_after_freeze: 6987


,rows_in,rows_kept,duplicate_timestamps,off_grid_timestamps,nonpositive_prices,inconsistent_ohlc,missing_minutes,largest_gap_minutes
series,,,,,,,,
BTCUSDT,3573284,3573284,0,0,0,0,0,1


## 12–17. Evaluation

12. Statistical evaluation: exact paired sign tests for after-cost Sharpe, with Holm correction across the two controls.
13. Economic evaluation: Sharpe, mean after-cost return, turnover, and drawdown under the frozen H5 budget and H4 costs.
14. Calibration: point-in-time unit ES, budget utilisation, and zero post-authorisation breaches for the candidate.
15. Ablation: RORAC versus equal-notional and versus volatility-targeted sizing on the identical parent decisions.
16. Robustness: every registered rolling origin and declared horizon must be complete.
17. Failure analysis: failed origins, unavailable risk estimates, denominator floors, caps, binds, and liquidation proxies remain visible.

In [3]:
from research.allocation import retained_sizer
from research.evaluate import evaluate_h6, h6_development_decision, h6_sizer_comparison

CONFIG = H6_CONFIG
EXPERIMENT_ID = CONFIG["experiment_id"]
RECORD_PATH = root / "research" / "registry" / "EXP-H06.json"
H5_RECORD_PATH = root / "research" / "registry" / "EXP-H05.json"
H4_RECORD_PATH = root / "research" / "registry" / "EXP-H04.json"
record = load_record(RECORD_PATH)
h5_record = load_record(H5_RECORD_PATH)
h4_record = load_record(H4_RECORD_PATH)
evaluation = evaluate_h6(bars, h5_record, h4_record, CONFIG)
if evaluation.sealed_access:
    raise RuntimeError("development notebook read the sealed test")
if evaluation.deferred:
    comparison = pd.DataFrame()
    allocation_summary_frame = pd.DataFrame()
    verdict = pd.Series(
        {
            "null_decision": "defer",
            "benchmark": "volatility_targeted",
            "reason": evaluation.defer_reason,
        }
    )
else:
    comparison = h6_sizer_comparison(evaluation.scores)
    verdict = h6_development_decision(comparison, CONFIG).iloc[0]
    allocation_summary_frame = allocation_summary(evaluation.scores)
GATE_PARENT_RECORD = h5_record
GATE_OBJECTS = manifest.objects
print("config_digest:", config_digest(CONFIG))
print("parent_rule:", evaluation.parent_rule)
print("budget_policy:", evaluation.budget_policy or h5_record.result)
print("default_sizer:", retained_sizer(record))
print("sealed_boundary:", "deferred" if evaluation.deferred else evaluation.boundary)
print("development_null_decision:", verdict.null_decision)
print("working_sizer:", verdict.benchmark)
print("failed_windows:", len(evaluation.failures))
if evaluation.deferred:
    print("defer_reason:", evaluation.defer_reason)

config_digest: bc90d7b0ebb7fdaddc3cce73edbef8c49be4428c509269cdc964ffeaaa51b9ae
parent_rule: first_passage_probability
budget_policy: unconstrained_h4
default_sizer: volatility_targeted
sealed_boundary: deferred
development_null_decision: defer
working_sizer: volatility_targeted
failed_windows: 0
defer_reason: H5 retained no ES budget; H6 cannot score sizing against unconstrained_h4


## How to read the scores

Sharpe is the pre-registered economic metric, and higher is better. The candidate must beat both controls at every horizon. Holm adjustment is applied to the two control comparisons after taking the weakest horizon in each comparison. A positive breach count fails the common-budget gate. Until H5 is sealed and retained, the tables below show the deferral instead of a sizing result.

In [ ]:
if evaluation.deferred:
    print(evaluation.defer_reason)
    display(evaluation.scores)
else:
    display(comparison.sort_values(["horizon_minutes", "control"]))
    display(allocation_summary_frame)
    if evaluation.failures:
        display(pd.DataFrame(evaluation.failures))

## Development economic plots

The first panel shows median after-cost Sharpe by sizer and horizon. The second shows denominator floors, unavailable ratios, and ES binds. Both panels stay empty while H6 is deferred.

In [ ]:
if evaluation.deferred:
    print(evaluation.defer_reason)
    display(evaluation.scores)
else:
    display(plot_allocation_sharpe(evaluation.scores))
    display(plot_allocation_activity(evaluation.scores))

## Risk calibration and adherence

The training RORAC normaliser and H5 ES scale are fitted separately at each origin using only labels available before validation. Constraint adherence must report zero post-authorisation budget breaches. Unavailable estimates fail closed to zero exposure and remain counted.

In [ ]:
if evaluation.deferred:
    print(evaluation.defer_reason)
    display(evaluation.diagnostics)
else:
    if evaluation.diagnostics.empty:
        raise RuntimeError("H6 produced no allocation diagnostics")
    if int(evaluation.diagnostics["constraint_breaches"].sum()) != 0:
        raise RuntimeError("H6 produced a post-authorisation ES budget breach")
    display(
        evaluation.diagnostics.groupby(["horizon_minutes", "sizer"])[
            [
                "training_rorac_normaliser",
                "training_positive_ratios",
                "training_expected_shortfall",
                "constraint_breaches",
                "denominator_floors",
                "unavailable_estimates",
                "gross_caps",
            ]
        ].median().reset_index()
    )

## Fitted allocation state

The table below isolates sizing activity. Every arm uses the same H4 direction and the same H5 budget. H6 can change size and the budget can reduce it; neither can flip the parent direction.

In [ ]:
activity_columns = [
    "horizon_minutes",
    "sizer",
    "es_binds",
    "denominator_floors",
    "unavailable_estimates",
    "gross_caps",
    "constraint_breaches",
    "liquidation_events",
    "median_budget_utilisation",
    "average_absolute_position",
]
if evaluation.deferred:
    print(evaluation.defer_reason)
    display(allocation_summary_frame)
else:
    display(allocation_summary_frame[activity_columns])

## 18. Result

This cell states the development answer only. It does not promote an allocator or read the sealed year.

In [ ]:
if verdict.null_decision == "reject":
    print(f"Answer: yes on development data; {verdict.benchmark} passes the registered gate.")
elif verdict.null_decision == "accept":
    print("Answer: no; keep volatility-targeted sizing as the default.")
else:
    print("Answer: not yet; H6 is deferred.")
print("Why:", verdict.reason)
print("This is development evidence. The sealed final year has not been read.")

## 19. Decision

Record the development working sizer without writing a sealed retain/reject decision. The registry is the lifecycle authority.

In [ ]:
updated_record = record_development_decision(
    RECORD_PATH,
    verdict,
    experiment_id=EXPERIMENT_ID,
    dataset_id=manifest.dataset_id,
    dataset_digest=manifest.digest,
    config=CONFIG,
    root=root,
)
display(asdict(updated_record))
blockers = confirmation_blockers(
    root=root,
    record=updated_record,
    parent_record=GATE_PARENT_RECORD,
    dataset_objects=GATE_OBJECTS,
)
print("confirmation_blockers:")
for blocker in blockers:
    print("-", blocker)

## Sealed confirmation

Leave disabled during development. Confirmation requires committed code, immutable dataset objects, the exact pre-registered digest, a development result of `rorac`, and one sealed-retained H5 budget.

In [ ]:
from research.evaluate import confirm_h6

RUN_SEALED_CONFIRMATION = False
if RUN_SEALED_CONFIRMATION:
    if blockers:
        raise RuntimeError("sealed confirmation is blocked: " + "; ".join(blockers))
    confirmation = confirm_h6(
        bars,
        updated_record,
        h5_record,
        h4_record,
        dataset_manifest=manifest,
        root=root,
        record_path=RECORD_PATH,
        config=CONFIG,
    )
    display(asdict(confirmation.record))
else:
    print("sealed confirmation disabled")

## 20. Reusable implementation extracted

`research.allocation` contains the floored barrier payoff, the unit-ES RORAC ratio, the training-only p90 normaliser, and the equal-notional, volatility-targeted, and RORAC sizers. `research.evaluate.evaluate_h6` and `confirm_h6` own walk-forward scoring and the sealed lifecycle. Volatility-targeted sizing remains the default until H6 is sealed and retained.

## 21. Next experiment

H7 (`09_ripple_information`, `EXP-H07`) tests whether order-book information changes a frozen Wave or Tide decision. It needs the L2 collector and does not reopen this sizing comparison.